In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Jul_16_20:06:48_Pacific_Daylight_Time_2025
Cuda compilation tools, release 13.0, V13.0.48
Build cuda_13.0.r13.0/compiler.36260728_0


In [3]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

When we researched how to approach object detection tasks, we found that Convolutional Neural Nets are the most widely used architecture in computer vision. So we decided to build a CNN-based model for our task.

When designing our CNN architecture, we reviewed how common computer vision models are implemented in PyTorch. A very common pattern in these models is to use a convolution layer followed by batch normalization and an activation function. 

To ensure our model follows these industry best practices, we created a reusable FireFeatureBlock block that combines Conv2d, BatchNorm2d, and LeakyReLU. We use this block as the fundamental building unit throughout our entire backbone.

![ConvBnLeakyReLU block](../docs/images/Firefeatureblock.png)

In [8]:
class FireFeatureBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1): # Padding = 1 means zero padding is applied to the edges
        super().__init__()
        self.block = nn.Sequential(
            # conv bias is mathematically cancelled out by BatchNorm's mean subtraction so we set it to False
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels), # Normalizes the data at each step preventing gradients from exploding or vanishing
            nn.LeakyReLU(0.1, inplace=True) # LeakyReLU is used instead of ReLU to prevent dead neurons (ReLU outputs 0 for negative values -> dead neurons)
        )

    def forward(self, x):
        return self.block(x)

In [9]:
# Testing the block to verify tensor shapes with a dummy input
dummy_input = torch.randn(1, 3, 416, 416)
test_block = FireFeatureBlock(in_channels=3, out_channels=32, stride=2) 

print("Input shape:", dummy_input.shape)
print("Output shape:", test_block(dummy_input).shape)

Input shape: torch.Size([1, 3, 416, 416])
Output shape: torch.Size([1, 32, 208, 208])
